In [ ]:
# ===========================================================
# Paper Section 1: Data representation & domain shift
# ===========================================================

# Cell 1 — Imports & config
import sys, os
sys.path.append(os.path.join(os.getcwd(), '../../'))

import pandas as pd
import numpy as np
import warnings
import pickle
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from cmcrameri import cm
import massbalancemachine as mbm

from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.dataset import (
    build_global_scalers_multi_source,
    estimate_global_bandwidths,
    compute_domain_shift,
)

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.plots.use_mbm_style()

BASE_DIR   = Path(cfg.dataPath) / path_cache / "CROSS_REGION"
REGION_GROUPS = {"CEU": ["FR", "IT_AT", "CH"]}
SOURCE_REGIONS = ["CH", "NOR", "ISL", "CEU"]
EXCLUDE_TARGETS = {"CAW", "ACA", "GRL"}

In [ ]:
df_stakes = pd.read_csv(
    os.path.join(cfg.dataPath, 'WGMS/raw/mass_balance_point.csv'))

df_RGIId = pd.read_csv(cfg.dataPath + 'WGMS/raw/glacier.csv')

# Create a mapping dictionary from id to rgi60_ids
id_to_rgi_map = dict(zip(df_RGIId['id'], df_RGIId['rgi60_ids']))

# Add the RGIId column to the filtered DataFrame using glacier_id instead of id
df_stakes['RGIId'] = df_stakes['glacier_id'].map(id_to_rgi_map)

# filter to non Nan RGIId
df_stakes = df_stakes[~df_stakes['RGIId'].isna()]
df_stakes.head()

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# --- aggregate per glacier (lat/lon + count) ---
df_glacier_counts = (
    df_stakes
    .groupby("RGIId")
    .agg(
        lat=("latitude", "mean"),
        lon=("longitude", "mean"),
        n_measurements=("id", "count"),
    )
    .reset_index()
    .dropna(subset=["lat", "lon"])
)

print(f"Total glaciers with stakes: {len(df_glacier_counts)}")
print(f"Total stake measurements:   {df_glacier_counts['n_measurements'].sum():,}")

# --- plot ---
fig = plt.figure(figsize=nature_figsize(cols=2, height_mm=100))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())

ax.set_global()
ax.add_feature(cfeature.LAND,       facecolor="#f0f0f0", linewidth=0)
ax.add_feature(cfeature.OCEAN,      facecolor="#d6e8f5", linewidth=0)
ax.add_feature(cfeature.COASTLINE,  linewidth=0.3, edgecolor="#999999")
ax.add_feature(cfeature.BORDERS,    linewidth=0.2, edgecolor="#cccccc")

# size scaled by number of measurements
sizes = np.clip(df_glacier_counts["n_measurements"], 1, 500)
sizes_scaled = 5 + 80 * (sizes - sizes.min()) / (sizes.max() - sizes.min())

sc = ax.scatter(
    df_glacier_counts["lon"],
    df_glacier_counts["lat"],
    s=sizes_scaled,
    # c=df_glacier_counts["n_measurements"],
    cmap=cm.batlow,
    alpha=0.7,
    linewidths=0.2,
    edgecolors="white",
    transform=ccrs.PlateCarree(),
    zorder=3,
)

# cbar = fig.colorbar(sc, ax=ax, orientation="horizontal", pad=0.05, fraction=0.03)
# cbar.set_label("Number of stake measurements", fontsize=NATURE_SPECS["font_min_pt"])
# cbar.ax.tick_params(labelsize=NATURE_SPECS["font_min_pt"])

# size legend
for n_val in [10, 100, 500]:
    s_val = 1 + 40 * (n_val - sizes.min()) / (sizes.max() - sizes.min())
    ax.scatter([], [], s=s_val, c="gray", alpha=0.7, label=f"{n_val}",
               transform=ccrs.PlateCarree())
ax.legend(
    title="Measurements",
    loc="lower left",
    fontsize=NATURE_SPECS["font_min_pt"],
    title_fontsize=NATURE_SPECS["font_min_pt"],
    frameon=True,
    framealpha=0.8,
)

ax.set_title(
    "Global glacier stake measurement locations (WGMS)",
    fontsize=NATURE_SPECS["font_max_pt"],
    pad=8,
)

plt.tight_layout()
plt.show()

In [ ]:
df_stakes.RGIId.nunique()

In [ ]:
# Cell 2 — Load stake data
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}

# Pool all stake data into a single df with SOURCE_CODE label
df_all_stakes = pd.concat(dfs.values(), ignore_index=True)
print("Total stake measurements:", len(df_all_stakes))
print("Regions:", df_all_stakes["SOURCE_CODE"].unique())
print(df_all_stakes.groupby("SOURCE_CODE").size().rename("n_measurements"))

In [ ]:
# Cell 3 — Load monthly datasets (needed for feature distributions)
MONTHLY_COLS = ['t2m', 'tp', 'slhf', 'sshf', 'ssrd', 'fal', 'str', 'ELEVATION_DIFFERENCE']
STATIC_COLS  = ['aspect', 'slope', 'svf']

paths = {
    'era5_climate_data': os.path.join(
        cfg.dataPath, path_ERA5_raw, "era5_monthly_averaged_data_EU_US_CANADA.nc"),
    'geopotential_data': os.path.join(
        cfg.dataPath, path_ERA5_raw, "era5_geopotential_pressure_EU_US_CANADA.nc"),
}

experiment_region_groups = {"CEU": ["FR", "IT_AT", "CH"]}
vois_climate      = ["t2m", "tp", "slhf", "sshf", "ssrd", "fal", "str"]
vois_topographical = ["aspect", "slope", "svf"]

res_xreg_by_source = {}
for src_region in SOURCE_REGIONS:
    res_xreg, _ = prepare_monthly_df_xreg_SOURCE_to_EU(
        cfg=cfg, dfs=dfs, paths=paths,
        vois_climate=vois_climate,
        vois_topographical=vois_topographical,
        run_flag=False,
        source_code=src_region,
        region_groups=experiment_region_groups,
    )
    res_xreg_by_source[src_region] = res_xreg

In [ ]:
# Cell 4 — Plot 1.1: Data coverage map (two panels)
df_glaciers = (df_all_stakes
    .groupby(["GLACIER", "SOURCE_CODE"])
    .agg(n_measurements=("POINT_BALANCE", "count"),
         lat=("POINT_LAT", "mean"),
         lon=("POINT_LON", "mean"))
    .reset_index())

REGION_DISPLAY = {
    "CH":    {"label": "Switzerland",   "color": "#2166ac"},
    "FR":    {"label": "France",        "color": "#4dac26"},
    "IT_AT": {"label": "Italy/Austria", "color": "#b8e186"},
    "NOR":   {"label": "Norway",        "color": "#d01c8b"},
    "ISL":   {"label": "Iceland",       "color": "#f1b6da"},
    "SJM":   {"label": "Svalbard",      "color": "#e66101"},
    "ALA":   {"label": "Alaska",        "color": "#998ec3"},
    "CAW":   {"label": "W. Canada/US",  "color": "#542788"},
    "ACA":   {"label": "Arctic Canada", "color": "#b2abd2"},
    "GRL":   {"label": "Greenland",     "color": "#fee0b6"},
}

EUROPE_CODES = ["CH", "FR", "IT_AT", "NOR", "ISL", "SJM"]
US_CODES     = ["ALA", "CAW", "ACA", "GRL"]

import cartopy.crs as ccrs
import cartopy.feature as cfeature

def _add_base_features(ax):
    ax.add_feature(cfeature.LAND,      facecolor="#f0f0f0", zorder=0)
    ax.add_feature(cfeature.OCEAN,     facecolor="#d6e8f5", zorder=0)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.4,       zorder=1)
    ax.add_feature(cfeature.BORDERS,   linewidth=0.2,       zorder=1)
    ax.gridlines(linewidth=0.3, color="gray", alpha=0.4, linestyle="--")

fig = plt.figure(figsize=(16, 6))

# --- Panel A: Europe + Iceland ---
ax_eu = fig.add_subplot(1, 2, 1, projection=ccrs.LambertConformal(
    central_longitude=10, central_latitude=55))
ax_eu.set_extent([-30, 35, 35, 82], crs=ccrs.PlateCarree())
_add_base_features(ax_eu)

for src_code in EUROPE_CODES:
    grp = df_glaciers[df_glaciers["SOURCE_CODE"] == src_code]
    if grp.empty:
        continue
    ax_eu.scatter(
        grp["lon"], grp["lat"],
        s=np.sqrt(grp["n_measurements"]) * 4,
        c=REGION_DISPLAY[src_code]["color"],
        alpha=0.8, zorder=3,
        transform=ccrs.PlateCarree(),
        label=REGION_DISPLAY[src_code]["label"],
    )
ax_eu.set_title("Europe & Iceland", fontsize=11)
ax_eu.legend(loc="lower left", fontsize=8, frameon=True,
             title="Region", title_fontsize=8)

# --- Panel B: North America + Arctic ---
ax_us = fig.add_subplot(1, 2, 2, projection=ccrs.LambertConformal(
    central_longitude=-100, central_latitude=60))
ax_us.set_extent([-170, -50, 40, 85], crs=ccrs.PlateCarree())
_add_base_features(ax_us)

for src_code in US_CODES:
    grp = df_glaciers[df_glaciers["SOURCE_CODE"] == src_code]
    if grp.empty:
        continue
    ax_us.scatter(
        grp["lon"], grp["lat"],
        s=np.sqrt(grp["n_measurements"]) * 4,
        c=REGION_DISPLAY[src_code]["color"],
        alpha=0.8, zorder=3,
        transform=ccrs.PlateCarree(),
        label=REGION_DISPLAY[src_code]["label"],
    )
ax_us.set_title("North America & Arctic", fontsize=11)
ax_us.legend(loc="lower left", fontsize=8, frameon=True,
             title="Region", title_fontsize=8)

# --- shared size legend ---
for n, label in [(10, "10"), (100, "100"), (500, "500")]:
    ax_us.scatter([], [], s=np.sqrt(n) * 4, c="gray", alpha=0.6,
                  label=f"{label} meas.")
ax_us.legend(loc="lower right", fontsize=7, frameon=True,
             title="# measurements", title_fontsize=7, ncol=1)

fig.suptitle("Glacier stake measurement coverage by region", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("figures/section1_coverage_map.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Cell 5 — Plot 1.2: Feature KDEs per region
# Use the CH-source monthly df (contains all regions in train+test)
df_ch = pd.concat([
    res_xreg_by_source["CH"]["df_train"],
    res_xreg_by_source["CH"]["df_test"],
], ignore_index=True)

# Individual source codes to show (exclude group codes)
INDIVIDUAL_CODES = ["CH", "FR", "IT_AT", "NOR", "ISL", "SJM", "ALA"]
PLOT_COLS = ["t2m", "tp", "ELEVATION_DIFFERENCE", "aspect", "slope", "svf"]
COL_LABELS = {
    "t2m":                "2m temperature (°C)",
    "tp":                 "Precipitation (m/month)",
    "ELEVATION_DIFFERENCE": "Elevation diff. (m)",
    "aspect":             "Aspect (°)",
    "slope":              "Slope (°)",
    "svf":                "Sky-view factor",
}

colors = {c: REGION_DISPLAY[c]["color"]
          for c in INDIVIDUAL_CODES if c in REGION_DISPLAY}

fig, axes = plt.subplots(2, 3, figsize=(13, 7), squeeze=False)

for idx, col in enumerate(PLOT_COLS):
    ax = axes[idx // 3][idx % 3]
    for src_code in INDIVIDUAL_CODES:
        vals = df_ch.loc[df_ch["SOURCE_CODE"] == src_code, col].dropna()
        if len(vals) < 10:
            continue
        vals.plot.kde(ax=ax, label=REGION_DISPLAY[src_code]["label"],
                      color=colors[src_code], linewidth=1.5)
    ax.set_xlabel(COL_LABELS[col], fontsize=9)
    ax.set_ylabel("Density", fontsize=9)
    ax.set_title(COL_LABELS[col], fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(color="#e0e0e0", linewidth=0.5)
    ax.set_axisbelow(True)

handles = [mpatches.Patch(color=colors[c],
           label=REGION_DISPLAY[c]["label"])
           for c in INDIVIDUAL_CODES if c in colors]
fig.legend(handles=handles, loc="lower center", ncol=len(INDIVIDUAL_CODES),
           frameon=False, fontsize=9, bbox_to_anchor=(0.5, -0.04))

fig.suptitle("Climate & topographic feature distributions by region", fontsize=13)
plt.tight_layout()
plt.savefig("figures/section1_feature_kdes.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Cell 6 — Load or compute domain shifts for heatmap
CLIMATE_COLS = ['t2m', 'tp', 'slhf', 'sshf', 'ssrd', 'fal', 'str', 'ELEVATION_DIFFERENCE']
TOPO_COLS    = ['aspect', 'slope', 'svf']
SHIFTS_CACHE = BASE_DIR / "domain_shifts_cache_basic.pkl"

with open(SHIFTS_CACHE, "rb") as f:
    cache = pickle.load(f)

scaler_m             = cache["scaler_m"]
scaler_s             = cache["scaler_s"]
blur_m               = cache["blur_m"]
blur_s               = cache["blur_s"]
blur_joint           = cache["blur_joint"]
all_shifts_by_source = cache["all_shifts_by_source"]
print(f"Loaded {sum(len(v) for v in all_shifts_by_source.values())} shift entries")

In [ ]:
# Cell 7 — Plot 1.3: Domain shift heatmap
# Build source x target matrix of Sinkhorn joint distance
all_keys = []
for src, shifts in all_shifts_by_source.items():
    for k in shifts:
        all_keys.append(k)

# extract unique sources and targets
sources = SOURCE_REGIONS
targets = sorted(set(
    k.split("_TO_")[1] for k in all_keys
    if not any(ex in k for ex in EXCLUDE_TARGETS)
))

matrix = pd.DataFrame(index=sources, columns=targets, dtype=float)

for src in sources:
    for tgt in targets:
        key = f"XREG_{src}_TO_{tgt}"
        if key in all_shifts_by_source.get(src, {}):
            matrix.loc[src, tgt] = all_shifts_by_source[src][key].get(
                "D_sinkhorn_joint", float("nan"))

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(
    matrix.astype(float),
    ax=ax,
    cmap="YlOrRd",
    annot=True, fmt=".2f",
    linewidths=0.5,
    cbar_kws={"label": "Sinkhorn joint distance"},
)
ax.set_xlabel("Target region", fontsize=10)
ax.set_ylabel("Source region", fontsize=10)
ax.set_title("Pairwise domain shift (Sinkhorn joint, true)", fontsize=12)
plt.tight_layout()
plt.savefig("figures/section1_shift_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()